<a href="https://colab.research.google.com/github/BuruhArloji/PythonDataScienceHandbook/blob/master/MT_Weekly_Forecast_2026_08_17_2026_08_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Trade Weekly Forecast — Daily to Weekly

Notebook end-to-end untuk forecast minggu 17–23 dan 24–30 Agustus 2026. Grain kerja: `DATE × Group × WH × Item Code`; output: `WEEK START × WH × Item Code`.


## 1. Configuration and reusable Python functions
Seluruh fungsi disimpan di file Python pendamping agar notebook tetap mudah diaudit dan dijalankan ulang.


In [1]:
!pip install -q pandas numpy openpyxl xlsxwriter plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import sys

BASE_DIR = Path("/content/drive/MyDrive/MT_Forecast")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File MT_Weekly_Forecast_Pipeline.py harus berada di BASE_DIR
PIPELINE_FILE = BASE_DIR / "MT_Weekly_Forecast_Pipeline.py"
assert PIPELINE_FILE.exists(), f"Pipeline tidak ditemukan: {PIPELINE_FILE}"

sys.path.insert(0, str(BASE_DIR))

import MT_Weekly_Forecast_Pipeline as fp

SALES_FILE = INPUT_DIR / "2026.08.12 Indonesia Sales Dashboard.xlsm"
MASTER_FILE = INPUT_DIR / "Master_Internal.xlsx"
PROMO_FILE = INPUT_DIR / "MT Promo Calendar Form.xlsx"

assert SALES_FILE.exists(), f"Sales tidak ditemukan: {SALES_FILE}"
assert MASTER_FILE.exists(), f"Master tidak ditemukan: {MASTER_FILE}"
assert PROMO_FILE.exists(), f"Promo tidak ditemukan: {PROMO_FILE}"

# Arahkan output pipeline ke Google Drive
fp.OUTPUT_DIR = OUTPUT_DIR
fp.OUTPUT_XLSX = OUTPUT_DIR / "MT_Weekly_Forecast_2026-08-17_2026-08-24.xlsx"
fp.OUTPUT_HTML = OUTPUT_DIR / "MT_Weekly_Forecast_2026-08-17_2026-08-24_Plotly.html"

print({
    "sales": SALES_FILE,
    "master": MASTER_FILE,
    "promo": PROMO_FILE,
    "output": OUTPUT_DIR,
})

AssertionError: Sales tidak ditemukan: /content/drive/MyDrive/MT_Forecast/input/2026.08.12 Indonesia Sales Dashboard.xlsm

## 2. Load and standardize 2025–2026 sales


In [ ]:
raw = fp.load_sales_sources()
raw.groupby('SOURCE_SHEET').agg(rows=('ROW_ID','size'), min_date=('DATE','min'), max_date=('DATE','max'), qty=('QTY_PACK','sum'))


## 3. NCC cleaning
Exact LIFO menggunakan WH + customer + item + price, lalu residual dicari pada pembelian customer yang sama dalam 30 hari sebelumnya.


In [ ]:
clean, ncc_allocations, ncc_unmatched, ncc_qc = fp.clean_ncc(raw)
ncc_qc


## 4. Filter Modern Trade and map official Group


In [ ]:
customer_master = fp.load_customer_master()
mt, customer_unmatched = fp.map_modern_trade(clean, customer_master)
product_dim = fp.build_product_dim(mt)
{'rows': len(mt), 'packs': mt[fp.QTY].sum(), 'groups': mt['GROUP'].nunique(), 'wh': mt['WH'].nunique(), 'sku': mt['ITEM_CODE'].nunique(), 'unmatched_customer': len(customer_unmatched)}


## 5. Normalize promo calendar
Channel tidak membatasi scope. Promo diubah menjadi Item Code × Date; overlap mekanisme menjadi MULTIPLE_PROMO.


In [ ]:
promo_source, promo_official, promo_episodes, promo_effect = fp.load_clean_promo()
{'source_rows': len(promo_source), 'official_daily_rows': len(promo_official), 'episodes': len(promo_episodes), 'effect_rows': len(promo_effect)}


## 6. Build complete daily panel


In [ ]:
panel, actual_cutoff = fp.build_daily_panel(mt, product_dim, promo_effect)
{'rows': len(panel), 'cutoff': actual_cutoff, 'series': panel[['GROUP','WH','ITEM_CODE']].drop_duplicates().shape[0]}


## 7. De-promote historical sales
Actual tetap disimpan. Pada promo-affected dates, baseline memakai nilai yang lebih rendah antara actual dan counterfactual no-promo agar proses de-promotion tidak menciptakan volume historis.


In [ ]:
baseline_history = fp.build_baseline_history(panel, actual_cutoff)
demand_class = fp.classify_demand(baseline_history)
demand_class['DEMAND_CLASS'].value_counts()


## 8. Rolling 14-day backtest and model selection


In [ ]:
backtest_detail, backtest_summary, model_selection = fp.rolling_backtest(baseline_history, demand_class, actual_cutoff)
model_selection.groupby('SELECTED_METHOD').agg(series=('ITEM_CODE','size'), median_wape=('BACKTEST_WAPE','median'))


## 9. Historical promo uplift
Uplift dihitung sebagai actual / counterfactual − 1 pada event window. Fallback: item → format → brand → category → global.


In [ ]:
uplift_events, uplift_summaries = fp.build_uplift_table(baseline_history, product_dim)
uplift_events[['ITEM_CODE','PROMO_EFFECT_CATEGORY','EVENT_START','EVENT_END','ACTUAL_QTY','COUNTERFACTUAL_QTY','UPLIFT_RATE']].head()


## 10. Daily forecast and weekly aggregation


In [ ]:
forecast_daily, forecast_weekly_group, forecast_weekly = fp.generate_forecast(panel, baseline_history, model_selection, demand_class, product_dim, uplift_summaries, actual_cutoff)
forecast_weekly.groupby('WEEK_START')[['BASELINE_PACK','PROMO_UPLIFT_PACK','FINAL_FC_PACK']].sum()


## 11. QA, reconciliation, and exports


In [ ]:
qc, qc_summary, model_benchmark = fp.build_qc(raw, clean, mt, customer_master, promo_source, promo_official, product_dim, panel, baseline_history, backtest_summary, forecast_daily, forecast_weekly, ncc_qc)
qc


In [ ]:
results = fp.export_all(locals())
results
